# Day 4 目标：整合 ResearchAgent v0.1 Demo

今天的核心任务不是继续加复杂功能，而是做阶段一收尾：

让 ResearchAgent v0.1 看起来像一个完整项目：能运行、能演示、能解释、能交给别人复现。

今天完成后，你的阶段一就正式结束。

# 一、Day 4 最终目标

你今天要交付的是一个 ResearchAgent v0.1：

输入科研问题
↓
规则 / LLM 分类
↓
LangGraph 路由
↓
进入对应任务节点
↓
输出分类来源、分类原因、处理结果

目前它还不是完整科研 Agent，但已经是一个可扩展的 Agent 骨架。

# 二、今天要完成的 5 件事
任务 1：整理最终项目结构

确认你的项目结构类似这样：

F:\ResearchAgent
├── .env
├── .env.example
├── .gitignore
├── README.md
├── requirements.txt
├── run_cli.py
├── notebooks
│   └── day1_langgraph_minimal.ipynb
└── src
    └── research_agent
        ├── __init__.py
        └── graph
            ├── __init__.py
            ├── state.py
            ├── nodes.py
            ├── router.py
            ├── workflow.py
            └── llm_classifier.py

今天你主要补齐这些文件：

.env.example
.gitignore
requirements.txt
README.md

# 三、任务 2：创建 .env.example

注意：不要把真实 API Key 放进 GitHub。

创建：

F:\ResearchAgent\.env.example

内容：

/# Whether to use LLM-based task classification.
/# true: use LLM classifier first, fallback to rule classifier if failed
/# false: use rule classifier only
USE_LLM_CLASSIFIER=false

/# OpenAI-compatible API config
OPENAI_API_KEY=your_api_key_here
OPENAI_BASE_URL=https://api.openai.com/v1
OPENAI_MODEL=gpt-4o-mini

你的真实 .env 保留在本地，不上传。

# 四、任务 3：创建 .gitignore

创建：

F:\ResearchAgent\.gitignore

内容：

/# Python
__pycache__/
*.py[cod]
*.pyo
*.pyd
*.egg-info/
.eggs/
dist/
build/

/# Virtual environments
.venv/
venv/
env/
.conda/

/# Environment variables
.env

/# Jupyter
.ipynb_checkpoints/

/# VSCode
.vscode/

/# Logs
*.log
logs/

/# OS
.DS_Store
Thumbs.db

这里最重要的是：

.env
.conda/

不要把密钥和本地环境传上去。

# 五、任务 4：创建 requirements.txt

在项目根目录创建：

F:\ResearchAgent\requirements.txt

内容：

langgraph
langchain
langchain-openai
python-dotenv
pydantic
rich
tqdm
ipykernel

你也可以用命令生成更精确版本：

cd F:\ResearchAgent
.\.conda\python.exe -m pip freeze > requirements.txt

不过对阶段一来说，我更推荐先用上面的简洁版 requirements，避免写进一堆系统相关依赖。

# 六、任务 5：优化 run_cli.py

今天可以稍微把命令行入口做得更像 Demo。

把 run_cli.py 改成这个版本：

from src.research_agent.graph.workflow import build_graph


def create_initial_state(query: str) -> dict:
    return {
        "query": query,
        "task_type": "",
        "result": "",
        "final_answer": "",
        "classifier_source": "",
        "route_reason": "",
    }


def run_agent(query: str) -> dict:
    graph = build_graph()
    result = graph.invoke(create_initial_state(query))
    return result


def print_welcome():
    print("=" * 60)
    print("ResearchAgent v0.1")
    print("A minimal LangGraph-based research assistant demo.")
    print("=" * 60)
    print("支持任务类型：")
    print("1. paper_question          论文问答 / 文献总结")
    print("2. experiment_analysis     实验结果 / 幻觉指标分析")
    print("3. dataset_recommendation  数据集推荐")
    print("4. report_generation       组会汇报 / PPT 文案")
    print("5. code_help               代码 / 环境 / 报错问题")
    print("6. general                 通用科研助手")
    print()
    print("输入 q / quit / exit 退出")


def main():
    graph = build_graph()

    print_welcome()

    while True:
        query = input("\n请输入你的科研问题：\n> ").strip()

        if query.lower() in ["q", "quit", "exit"]:
            print("已退出。")
            break

        if not query:
            print("输入不能为空，请重新输入。")
            continue

        result = graph.invoke(create_initial_state(query))

        print("\n===== Agent 输出 =====")
        print(result["final_answer"])


if __name__ == "__main__":
    main()

这个版本有两个小优化：

1. graph 只 build 一次，不在每次输入时重复 build
2. 把 initial_state 单独封装，结构更清楚

这就是工程化的小味道了。

# 七、检查 final_answer_node

建议你的 nodes.py 里最终回答节点保持这个版本：

def final_answer_node(state: AgentState) -> dict:
    final_answer = f"""
任务类型：{state["task_type"]}
分类来源：{state["classifier_source"]}
分类原因：{state["route_reason"]}
处理结果：{state["result"]}
""".strip()

    return {
        "final_answer": final_answer
    }

这样演示时别人能看到：

这个问题被分成了什么类型
是规则分类还是 LLM 分类
为什么这么分
路由后进入了哪个能力模块

这比只输出一句 mock result 更像 Agent Demo。

# 八、测试阶段一完整功能

运行：

cd F:\ResearchAgent
.\.conda\python.exe run_cli.py

然后测试这 6 条：

请帮我总结一篇多模态偏见论文

预期：

任务类型：paper_question
...
处理结果：这是论文问答任务，后续会接入论文 RAG。
请帮我分析 coco_val_n300_g1 的幻觉风险

预期：

任务类型：experiment_analysis
...
处理结果：这是实验分析任务，后续会接入 CSV / JSONL 分析工具。
请推荐适合做幻觉评估的数据集

预期：

任务类型：dataset_recommendation
...
处理结果：这是数据集推荐任务，后续会接入数据集资料库。
帮我生成组会汇报文本

预期：

任务类型：report_generation
...
处理结果：这是汇报生成任务，后续会接入 Report Writer。
ModuleNotFoundError: No module named langgraph 怎么解决

预期：

任务类型：code_help
...
处理结果：这是代码辅助任务，后续会接入代码解释与修改工具。
我今天应该怎么安排科研任务

预期：

任务类型：general
...
处理结果：这是通用科研助手任务。

# 九、写 README.md

创建：

F:\ResearchAgent\README.md

先用这个版本：

# ResearchAgent v0.1

ResearchAgent is a minimal LangGraph-based research assistant demo.

The current version focuses on building a clean and extensible Agent workflow skeleton. It supports task classification, conditional routing, and mock task nodes for different research scenarios.

## Features

- LangGraph-based workflow
- Shared AgentState
- Rule-based task classification
- Optional LLM-based task classification
- Fallback from LLM classifier to rule classifier
- Conditional routing to different task nodes
- CLI demo

## Supported Task Types

| Task Type | Description |
|---|---|
| `paper_question` | Paper Q&A, literature summary, paper comparison |
| `experiment_analysis` | Experiment result analysis, hallucination metric interpretation |
| `dataset_recommendation` | Dataset recommendation and dataset selection |
| `report_generation` | Group meeting report, PPT text, progress summary |
| `code_help` | Code explanation, bug fixing, environment issues |
| `general` | General research assistant task |

## Project Structure

```text
ResearchAgent
├── run_cli.py
├── requirements.txt
├── .env.example
└── src
    └── research_agent
        └── graph
            ├── state.py
            ├── nodes.py
            ├── router.py
            ├── workflow.py
            └── llm_classifier.py
Installation
conda create --prefix ./.conda python=3.11 -y
conda activate ./.conda
pip install -r requirements.txt

On Windows PowerShell, you can also run the local environment directly:

.\.conda\python.exe run_cli.py
Environment Variables

Copy .env.example to .env:

cp .env.example .env

For Windows PowerShell:

Copy-Item .env.example .env

Example:

USE_LLM_CLASSIFIER=false
OPENAI_API_KEY=your_api_key_here
OPENAI_BASE_URL=https://api.openai.com/v1
OPENAI_MODEL=gpt-4o-mini

If USE_LLM_CLASSIFIER=false, the system uses rule-based classification only.

If USE_LLM_CLASSIFIER=true, the system first tries LLM-based classification and falls back to rule-based classification if the LLM call fails.

Run
python run_cli.py

Or on Windows:

.\.conda\python.exe run_cli.py
Example Inputs
请帮我总结一篇多模态偏见论文
请帮我分析 coco_val_n300_g1 的幻觉风险
请推荐适合做幻觉评估的数据集
帮我生成组会汇报文本
ModuleNotFoundError: No module named langgraph 怎么解决
我今天应该怎么安排科研任务
Current Workflow
User Query
↓
classify_task
↓
route_task
├── paper_node
├── experiment_node
├── dataset_node
├── report_node
├── code_node
└── general_node
↓
final_answer
↓
END
Version

v0.1

This version is a minimal workflow skeleton. It does not yet include RAG, real experiment file analysis, paper retrieval, or report generation.


---

# 十、可选：创建一个快速测试脚本

如果你愿意，可以新增：

```text
F:\ResearchAgent\test_demo.py

内容：

from run_cli import run_agent


TEST_QUERIES = [
    "请帮我总结一篇多模态偏见论文",
    "请帮我分析 coco_val_n300_g1 的幻觉风险",
    "请推荐适合做幻觉评估的数据集",
    "帮我生成组会汇报文本",
    "ModuleNotFoundError: No module named langgraph 怎么解决",
    "我今天应该怎么安排科研任务",
]


def main():
    for query in TEST_QUERIES:
        result = run_agent(query)
        print("=" * 60)
        print("用户输入：", query)
        print(result["final_answer"])


if __name__ == "__main__":
    main()

运行：

.\.conda\python.exe test_demo.py

这个脚本的好处是：
你不用每次手动输入 6 条测试语句，一键验收。

# 十、可选：创建一个快速测试脚本

如果你愿意，可以新增：

```text
F:\ResearchAgent\test_demo.py

内容：

from run_cli import run_agent


TEST_QUERIES = [
    "请帮我总结一篇多模态偏见论文",
    "请帮我分析 coco_val_n300_g1 的幻觉风险",
    "请推荐适合做幻觉评估的数据集",
    "帮我生成组会汇报文本",
    "ModuleNotFoundError: No module named langgraph 怎么解决",
    "我今天应该怎么安排科研任务",
]


def main():
    for query in TEST_QUERIES:
        result = run_agent(query)
        print("=" * 60)
        print("用户输入：", query)
        print(result["final_answer"])


if __name__ == "__main__":
    main()

运行：

.\.conda\python.exe test_demo.py

这个脚本的好处是：
你不用每次手动输入 6 条测试语句，一键验收。

# 十一、可选：生成项目树

在 PowerShell 里运行：

tree /F

如果输出太多，可以先不用管 .conda，因为它会很大。

更推荐只看核心目录：

tree src /F

你应该看到：

src
└── research_agent
    ├── __init__.py
    └── graph
        ├── __init__.py
        ├── llm_classifier.py
        ├── nodes.py
        ├── router.py
        ├── state.py
        └── workflow.py

# 十二、Day 4 最终验收标准

今天你完成以下内容，就算阶段一正式完成：

1. run_cli.py 可以稳定运行
2. 6 种任务类型都能正确路由
3. USE_LLM_CLASSIFIER=false 时规则分类稳定运行
4. USE_LLM_CLASSIFIER=true 时 LLM 分类可用，失败也能 fallback
5. 新增 .env.example
6. 新增 .gitignore
7. 新增 requirements.txt
8. 新增 README.md
9. 能向别人解释项目结构和工作流

# 十三、你今天要能讲出来的话

阶段一结束后，你应该能这样介绍你的项目：

我做了一个基于 LangGraph 的科研 Agent v0.1 骨架。

它使用 AgentState 维护任务执行过程中的共享状态，通过 classify_task 节点识别用户问题类型，再通过 conditional edge 路由到不同任务节点，包括论文问答、实验分析、数据集推荐、汇报生成、代码帮助和通用科研助手。

目前分类节点支持规则分类和可选的 LLM 分类，并且 LLM 失败时会自动回退到规则分类，保证系统不会因为模型调用失败而中断。

这个版本还没有接入真实 RAG、实验文件分析和报告生成工具，但已经完成了 Agent 的核心工作流结构，后续可以在各个节点中逐步接入真实能力。

这段你以后可以直接改进成 GitHub README、项目介绍，甚至简历项目描述。

# 十四、阶段一完成后的下一步

Day 4 完成后，你先不要急着做大而全的东西。
阶段二我建议进入：

Paper RAG 最小版本：让 paper_node 能读本地 markdown / txt 文献笔记，并回答问题。

但今天别做阶段二。

今天只把 v0.1 整理干净。

一句话总结 Day 4：

不是加功能，而是把阶段一成果整理成一个能展示、能复现、能继续扩展的 ResearchAgent v0.1。